Multi-level indexing allows you to store data on multiple dimensions

In [1]:
import pandas as pd
import numpy as np

### Creating a Multi-Level Index from Tuples

In [6]:
index = pd.MultiIndex.from_tuples(
    [('A', 2020), ('A', 2021), ('B', 2020), ('B', 2021)],
    names=['Category', 'Year'])

df = pd.DataFrame({'Value': [10, 15, 20, 25]}, index=index)

print(df)

               Value
Category Year       
A        2020     10
         2021     15
B        2020     20
         2021     25


In [13]:
# Flatten the MultiIndex
df_reset = df.reset_index()
df_reset

,Category,Year,Value
0,A,2020,10
1,A,2021,15
2,B,2020,20
3,B,2021,25


In [14]:
df_set = df_reset.set_index(['Category', 'Year'])
df_set

Value
Category Year       
A        2020     10
         2021     15
B        2020     20
         2021     25

In [21]:
index = pd.MultiIndex.from_product(
    [['Product A', 'Product B'], ['Store X', 'Store Y']],
    names=['Product', 'Store']
)

columns = pd.Index(['Jan', 'Feb'], name='Month')

df_3d = pd.DataFrame([[100, 110], [120, 115], [90, 105], [95, 100]], 
                  index=index, columns=columns)
df_3d

Month              Jan  Feb
Product   Store            
Product A Store X  100  110
          Store Y  120  115
Product B Store X   90  105
          Store Y   95  100

### Selecting Data with Multi-Level Indexing

In [8]:
df.loc['A']       

,Value
Year,
2020,10
2021,15


In [9]:
 # Returns all years for Category A
df.loc[('A', 2021)]       # Returns a specific row

Value    15
Name: (A, 2021), dtype: int64

The .xs() method allows for quick selection based on a specific index level.

In [12]:
df.xs('A')

,Value
Year,
2020,10
2021,15


In [ ]:
df.xs(2020, level='Year')

,M,F
Day,,
Day 1,560,565
Day 2,516,622


### Stacking and unstacking

In [18]:
df

Value
Category Year       
A        2020     10
         2021     15
B        2020     20
         2021     25

In [20]:
#Unstack: convert row index to columns
df_unstacked = df.unstack(level='Year')
df_unstacked

Value     
Year      2020 2021
Category           
A           10   15
B           20   25

### Creating Multi-Level Indexes

### Nested Categories

In [27]:
#flat
df_flat = pd.DataFrame({
    'Product': ['X', 'Y'],
    'Revenue_Jan': [200, 220],
    'Revenue_Feb': [210, 230],
    'Cost_Jan': [80, 90],
    'Cost_Feb': [85, 95]
})

df_flat

,Product,Revenue_Jan,Revenue_Feb,Cost_Jan,Cost_Feb
0,X,200,210,80,85
1,Y,220,230,90,95


In [28]:
columns = pd.MultiIndex.from_tuples([
    ('Revenue', 'Jan'), ('Revenue', 'Feb'),
    ('Cost', 'Jan'), ('Cost', 'Feb')
], names=['Metric', 'Month'])

df_multi = pd.DataFrame([[200, 210, 80, 85], [220, 230, 90, 95]], 
                        index=['X', 'Y'], columns=columns)

df_multi

Metric Revenue      Cost    
Month      Jan  Feb  Jan Feb
X          200  210   80  85
Y          220  230   90  95

In [29]:
df_multi['Revenue'] - df_multi['Cost']

Month,Jan,Feb
X,120,125
Y,130,135


### Relation to Groupby

In [22]:
# Flat form
df_flat = pd.DataFrame({
    'Store': ['A', 'A', 'B', 'B'],
    'Date': ['2023-01-01', '2023-01-02', '2023-01-01', '2023-01-02'],
    'Sales': [100, 110, 90, 95]
})

df_flat

,Store,Date,Sales
0,A,2023-01-01,100
1,A,2023-01-02,110
2,B,2023-01-01,90
3,B,2023-01-02,95


In [23]:
# Hierarchical format
df_multi = df_flat.set_index(['Store', 'Date'])
df_multi

Sales
Store Date             
A     2023-01-01    100
      2023-01-02    110
B     2023-01-01     90
      2023-01-02     95

In [24]:
df_multi.groupby(level='Store').mean()

,Sales
Store,
A,105.0
B,92.5


### Multi-level indexing and stacking

In [30]:
# Step 1: Start with a flat DataFrame

df = pd.DataFrame({
    'Region': ['North', 'South'],
    'Product': ['A', 'B'],
    'Jan_Sales': [100, 150],
    'Feb_Sales': [110, 160],
    'Jan_Cost': [60, 80],
    'Feb_Cost': [65, 85]
})

# Flat structure
print("Original flat DataFrame:\n", df)

Original flat DataFrame:
   Region Product  Jan_Sales  Feb_Sales  Jan_Cost  Feb_Cost
0  North       A        100        110        60        65
1  South       B        150        160        80        85


In [31]:
# Step 2: Set a MultiIndex on rows, then stack columns
df_indexed = df.set_index(['Region', 'Product'])
df_indexed

,,Jan_Sales,Feb_Sales,Jan_Cost,Feb_Cost
Region,Product,,,,
North,A,100,110,60,65
South,B,150,160,80,85


In [32]:
# Stack moves column values into rows, creating a MultiIndex
df_stacked = df_indexed.stack()
print("\nStacked format (MultiIndex on rows):\n", df_stacked)


Stacked format (MultiIndex on rows):
 Region  Product           
North   A        Jan_Sales    100
                 Feb_Sales    110
                 Jan_Cost      60
                 Feb_Cost      65
South   B        Jan_Sales    150
                 Feb_Sales    160
                 Jan_Cost      80
                 Feb_Cost      85
dtype: int64


In [33]:
# Step 3: Split the stacked index into 'Month' and 'Metric'
df_stacked = df_stacked.rename('Value').reset_index()
df_stacked

,Region,Product,level_2,Value
0,North,A,Jan_Sales,100
1,North,A,Feb_Sales,110
2,North,A,Jan_Cost,60
3,North,A,Feb_Cost,65
4,South,B,Jan_Sales,150
5,South,B,Feb_Sales,160
6,South,B,Jan_Cost,80
7,South,B,Feb_Cost,85


In [34]:
# Split 'Jan_Sales' → 'Jan', 'Sales' using regex
df_stacked[['Month', 'Metric']] = df_stacked['level_2'].str.extract(r'(\w+)_([A-Za-z]+)')
df_stacked = df_stacked.drop(columns='level_2')
print("\nExploded stacked format with Month and Metric:\n", df_stacked)


Exploded stacked format with Month and Metric:
   Region Product  Value Month Metric
0  North       A    100   Jan  Sales
1  North       A    110   Feb  Sales
2  North       A     60   Jan   Cost
3  North       A     65   Feb   Cost
4  South       B    150   Jan  Sales
5  South       B    160   Feb  Sales
6  South       B     80   Jan   Cost
7  South       B     85   Feb   Cost


In [35]:
# Step 4: Pivot into tidy long format
df_long = df_stacked.pivot(index=['Region', 'Product', 'Month'], 
                           columns='Metric', 
                           values='Value').reset_index()
print("\nTidy long format with MultiIndex-friendly structure:\n", df_long)


Tidy long format with MultiIndex-friendly structure:
 Metric Region Product Month  Cost  Sales
0       North       A   Feb    65    110
1       North       A   Jan    60    100
2       South       B   Feb    85    160
3       South       B   Jan    80    150


In [36]:
# Step 5: Pivot again to wide format using MultiIndex on columns
# Set MultiIndex again for clean unstacking
df_long_indexed = df_long.set_index(['Region', 'Product', 'Month'])
df_wide = df_long_indexed.unstack(level='Month')
print("\nFinal wide format with MultiIndex on columns:\n", df_wide)


Final wide format with MultiIndex on columns:
 Metric         Cost     Sales     
Month           Feb Jan   Feb  Jan
Region Product                    
North  A         65  60   110  100
South  B         85  80   160  150


In [ ]:
# Optional: Inspect the column MultiIndex
print("\nColumn MultiIndex levels:\n", df_wide.columns)


Column MultiIndex levels:
 MultiIndex([( 'Cost', 'Feb'),
            ( 'Cost', 'Jan'),
            ('Sales', 'Feb'),
            ('Sales', 'Jan')],
           names=['Metric', 'Month'])
